# kluster.ai

[kluster.ai](https://kluster.ai) is a inference service that provides access to a variety of high-performance LLMs including Meta's Llama 3.1 and Llama 3.3 models. This notebook goes over how to use LangChain with kluster.ai for chat models.

## Set the Environment API Key
Make sure to get your API key from kluster.ai. You need to [sign up](https://kluster.ai/auth/signup) and create a new API token from your dashboard.

kluster.ai offers a free tier with generous credits to test their models without requiring a credit card.

In [ ]:
# get a new token from your kluster.ai dashboard if not already set

import os
from getpass import getpass

from langchain_community.chat_models import ChatKlusterAi
from langchain_core.messages import HumanMessage

# Only prompt for API token if not already set in environment
if "KLUSTERAI_API_KEY" not in os.environ:
    print("Please enter your kluster.ai API token:")
    KLUSTERAI_API_TOKEN = getpass()
    os.environ["KLUSTERAI_API_KEY"] = KLUSTERAI_API_TOKEN
else:
    print("Using existing KLUSTERAI_API_KEY from environment")

chat = ChatKlusterAi(model="klusterai/Meta-Llama-3.1-8B-Instruct-Turbo")

messages = [
    HumanMessage(
        content="Translate this sentence from English to French. I love programming."
    )
]
chat.invoke(messages)

## Using system messages

You can also use system messages to provide context and instructions to the model:

In [ ]:
from langchain_core.messages import SystemMessage

messages = [
    SystemMessage(
        content="You are a professional translator for English to French. Translate directly without explanations."
    ),
    HumanMessage(content="I love programming."),
]

chat.invoke(messages)

## Controlling model parameters

You can control various parameters such as temperature to adjust the model's creativity level:

In [ ]:
# Using a more deterministic output with lower temperature
precise_chat = ChatKlusterAi(
    model="klusterai/Meta-Llama-3.1-8B-Instruct-Turbo", temperature=0.1
)

precise_chat.invoke(messages)

## Async and streaming functionality

ChatKlusterAi supports both async calls and streaming functionality:

In [ ]:
from langchain_core.callbacks import StreamingStdOutCallbackHandler

In [ ]:
# Example of async generation
simple_messages = [
    HumanMessage(
        content="Translate this sentence from English to French. I love programming."
    )
]
await chat.agenerate([simple_messages])

In [ ]:
# Example of streaming functionality
streaming_chat = ChatKlusterAi(
    model="klusterai/Meta-Llama-3.1-8B-Instruct-Turbo",
    streaming=True,
    verbose=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)
streaming_chat.invoke(simple_messages)

## Using different models

kluster.ai offers various models with different capabilities and sizes. Here's how to use different models:

In [ ]:
# Using the larger 405B parameter model
large_model_chat = ChatKlusterAi(
    model="klusterai/Meta-Llama-3.1-405B-Instruct-Turbo",
    temperature=0.2,
)

large_model_chat.invoke(simple_messages)

## Tool Calling

kluster.ai supports tool calling functionality with models like Meta-Llama-3.1-405B and Meta-Llama-3.3-70B. This lets you define tools that the model can call to perform specific actions.

For a complete list of models that support tool calling, please refer to the [kluster.ai documentation](https://kluster.ai).

In [ ]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field

print("Tool definition example:")


# Define a simple tool using the @tool decorator
class GetWeather(BaseModel):
    """Get the current weather in a given location"""

    location: str = Field(..., description="The city and state, e.g. San Francisco, CA")


# Define a tool using Pydantic for more complex parameters
class SearchQuery(BaseModel):
    """Search for information on a given topic."""

    query: str = Field(..., description="The search query")
    max_results: int = Field(5, description="Maximum number of results to return")

In [ ]:
# Set up the chat model with tool capabilities
tool_capable_model = ChatKlusterAi(
    model="klusterai/Meta-Llama-3.1-405B-Instruct-Turbo",
    temperature=0.1,
)

# Bind the tools to the model
llm_with_tools = tool_capable_model.bind_tools([GetWeather])

# Create a message that will trigger tool usage
weather_query = "What's the weather in San Francisco?"
print(f"Prompt: {weather_query}")

# Send the query to the model
response = llm_with_tools.invoke(weather_query)
response

In [ ]:
# Examine the tool calls
print(f"Tool calls: {response.tool_calls}\n")

# Example of processing the tool call
print("Example of processing tool calls and responding:")
if response.tool_calls:
    for tool_call in response.tool_calls:
        if tool_call["name"] == "get_weather":
            location = tool_call["args"]["location"]
            weather_result = get_weather(location)
            print(weather_result)

## Using PromptTemplates with ChatKlusterAi

You can combine ChatKlusterAi with LangChain's PromptTemplates for more structured interactions:

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Create a chat prompt template
prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a professional translator from {source_language} to {target_language}.",
        ),
        ("human", "{text_to_translate}"),
    ]
)

# Create our chat chain
translation_chain = prompt_template | chat

# Run the chain
response = translation_chain.invoke(
    {
        "source_language": "English",
        "target_language": "French",
        "text_to_translate": "I like programming in Python.",
    }
)

print(f"Response: {response.content}")

## Batch Processing with kluster.ai

For handling larger workloads, kluster.ai supports batch processing. Here's a simple example:

In [ ]:
# Example of processing multiple translation requests
texts_to_translate = [
    "I love programming.",
    "The weather is nice today.",
    "Can you help me with this problem?",
]

print("Processing batch of translation requests...\n")

# Process each request
for text in texts_to_translate:
    system_msg = SystemMessage(
        content="You are a professional French translator. Translate directly without explanations."
    )
    human_msg = HumanMessage(content=text)
    response = chat.invoke([system_msg, human_msg])

    print(f"Input: {text}")
    print(f"Translation: {response.content}\n")

## Error Handling

When working with any API, proper error handling is important:

In [ ]:
import time

print("Example of error handling:")
chat = ChatKlusterAi(
    model="klusterai/Meta-Llama-3.1-8B-Instruct-Turbo",
    max_tokens=1,
)


def safe_translation(text, max_retries=3):
    """Safely translate text with retries"""
    retry_count = 0

    while retry_count < max_retries:
        try:
            system_msg = SystemMessage(
                content="You are a professional French translator. Translate directly."
            )
            human_msg = HumanMessage(content=text)
            response = chat.invoke([system_msg, human_msg])
            return f"Translation successful: {response.content}"
        except Exception as e:
            retry_count += 1
            if retry_count >= max_retries:
                return f"Failed after {max_retries} attempts. Error: {str(e)}"
            print(f"Attempt {retry_count} failed. Retrying in 2 seconds...")
            time.sleep(2)


# Try to translate with error handling
result = safe_translation("I enjoy learning new technologies.")
print(result)